In [1]:
import pandas as pd
import h3

df = pd.read_parquet("./data/final_df.parquet", engine="pyarrow")

In [2]:
df2=df[df["rideStatus"]=="ON_PICKUP"]

In [3]:
cols = ["rid", "ts", "trip_start_time"]   # example
df2 = df2[cols]

In [7]:
df2["ts"] = pd.to_datetime(df2["ts"]).values.astype("int64")/1e9
df2["trip_start_time"] = pd.to_datetime(df2["trip_start_time"]).values.astype("int64")/1e9
df2["pickup_delay"] = df2["trip_start_time"] - df2["ts"]

In [8]:
df2["pickup_delay"].describe()

count    831202.000000
mean        210.943378
std         170.790147
min          -1.218000
25%          89.994000
50%         169.850500
75%         283.143000
max        2277.232000
Name: pickup_delay, dtype: float64

In [2]:
df.columns

Index(['rid', 'driver_id', 'ts', 'lat', 'lon', 'rideStatus', 'rider_id',
       'status', 'trip_start_lat', 'trip_start_lon', 'trip_end_lat',
       'trip_end_lon', 'trip_start_time', 'trip_end_time'],
      dtype='object')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4637888 entries, 0 to 4637887
Data columns (total 14 columns):
 #   Column           Dtype         
---  ------           -----         
 0   rid              object        
 1   driver_id        object        
 2   ts               datetime64[ns]
 3   lat              object        
 4   lon              object        
 5   rideStatus       object        
 6   rider_id         object        
 7   status           object        
 8   trip_start_lat   float64       
 9   trip_start_lon   float64       
 10  trip_end_lat     float64       
 11  trip_end_lon     float64       
 12  trip_start_time  datetime64[ns]
 13  trip_end_time    datetime64[ns]
dtypes: datetime64[ns](3), float64(4), object(7)
memory usage: 495.4+ MB


In [4]:
df2 = df.drop_duplicates(subset="rid", keep="first").reset_index(drop=True)

In [5]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14391 entries, 0 to 14390
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   rid              14391 non-null  object        
 1   driver_id        14391 non-null  object        
 2   ts               14391 non-null  datetime64[ns]
 3   lat              14391 non-null  object        
 4   lon              14391 non-null  object        
 5   rideStatus       14391 non-null  object        
 6   rider_id         14391 non-null  object        
 7   status           14391 non-null  object        
 8   trip_start_lat   14391 non-null  float64       
 9   trip_start_lon   14391 non-null  float64       
 10  trip_end_lat     14391 non-null  float64       
 11  trip_end_lon     14391 non-null  float64       
 12  trip_start_time  14391 non-null  datetime64[ns]
 13  trip_end_time    14391 non-null  datetime64[ns]
dtypes: datetime64[ns](3), float64(4), obje

In [6]:
cols = ["rid", "ts", "lat", "lon", "rideStatus"]   # example
df = df[cols]

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4637888 entries, 0 to 4637887
Data columns (total 5 columns):
 #   Column      Dtype         
---  ------      -----         
 0   rid         object        
 1   ts          datetime64[ns]
 2   lat         object        
 3   lon         object        
 4   rideStatus  object        
dtypes: datetime64[ns](1), object(4)
memory usage: 176.9+ MB


In [8]:
df2.to_parquet(
    "./data/uniquerid.parquet",
    engine="pyarrow",      # faster, preferred
    compression="snappy",  # good balance
    index=False
)

In [9]:
df.to_parquet(
    "./data/rids.parquet",
    engine="pyarrow",      # faster, preferred
    compression="snappy",  # good balance
    index=False
)

In [1]:
import pandas as pd
import h3

df = pd.read_parquet("./data/rids.parquet", engine="pyarrow")

In [2]:
df2 = pd.read_parquet("./data/uniquerid.parquet", engine="pyarrow")

In [3]:
# Convert to float (invalid parsing -> NaN)
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
df["lon"] = pd.to_numeric(df["lon"], errors="coerce")

# Optional: drop invalid rows
df = df.dropna(subset=["lat", "lon"])

# Generate H3 index
df["curr_hex"] = [
    h3.latlng_to_cell(lat, lon, 9)
    for lat, lon in zip(df["lat"], df["lon"])
]

# --- also convert lat/lon in df2 to float (they may be stored as object) ---
df2["lat"] = pd.to_numeric(df2["lat"], errors="coerce")
df2["lon"] = pd.to_numeric(df2["lon"], errors="coerce")

In [4]:

df['ts'] = pd.to_datetime(df['ts'])
df = df.sort_values('ts').reset_index(drop=True)

df2['ts']            = pd.to_datetime(df2['ts'])
df2['trip_end_time'] = pd.to_datetime(df2['trip_end_time'])

# Build a dict:  rid -> {trip_end_time, trip_end_lat, trip_end_lon, ...}
# This replaces the columns that used to be directly in df (nypool2.1).
ride_meta = {}
for _, r in df2.iterrows():
    ride_meta[r['rid']] = {
        'trip_end_time':  r['trip_end_time'],
        'trip_end_lat':   r['trip_end_lat'],
        'trip_end_lon':   r['trip_end_lon'],
        'trip_start_lat': r['trip_start_lat'],
        'trip_start_lon': r['trip_start_lon'],
    }

print(f"df shape : {df.shape}")
print(f"df2 shape: {df2.shape}")
print(f"ride_meta keys: {len(ride_meta)}")
print(f"Time range: {df['ts'].min()} → {df['ts'].max()}")


df shape : (4637888, 6)
df2 shape: (14391, 14)
ride_meta keys: 14391
Time range: 2025-08-01 15:00:00.030000 → 2025-08-01 20:59:59.942000


In [5]:

# 1. Sort df by rid then ts so groupby preserves chronological order
df_sorted = df.sort_values(['rid', 'ts'])

df_sorted = df_sorted[df_sorted["rideStatus"]=="ON_RIDE"]

print(len(df_sorted))

# 2. Group and aggregate
def _unique_ordered(series):
    """Return unique values preserving first-seen order."""
    seen = set()
    result = []
    for v in series:
        if v not in seen:
            seen.add(v)
            result.append(v)
    return result

journey = (
    df_sorted
    .groupby('rid', sort=False)
    .apply(
        lambda g: pd.Series({
            'journey_coords': list(zip(g['lat'], g['lon'])),
            'journey_hexes':  _unique_ordered(g['curr_hex']),
        })
    )
    .reset_index()
)

# 3. Merge into df2
df2 = df2.merge(journey, on='rid', how='left')

print(f"df2 shape after merge: {df2.shape}")
print(f"Columns: {df2.columns.tolist()}")
print(f"\nSample journey_coords (first ride): {df2['journey_coords'].iloc[0][:5]} ...")
print(f"Sample journey_hexes  (first ride): {df2['journey_hexes'].iloc[0][:5]} ...")
print(f"\njourney_coords lengths: min={df2['journey_coords'].apply(len).min()}, "
      f"max={df2['journey_coords'].apply(len).max()}, "
      f"mean={df2['journey_coords'].apply(len).mean():.1f}")
print(f"journey_hexes  lengths: min={df2['journey_hexes'].apply(len).min()}, "
      f"max={df2['journey_hexes'].apply(len).max()}, "
      f"mean={df2['journey_hexes'].apply(len).mean():.1f}")


3806686
df2 shape after merge: (14391, 16)
Columns: ['rid', 'driver_id', 'ts', 'lat', 'lon', 'rideStatus', 'rider_id', 'status', 'trip_start_lat', 'trip_start_lon', 'trip_end_lat', 'trip_end_lon', 'trip_start_time', 'trip_end_time', 'journey_coords', 'journey_hexes']

Sample journey_coords (first ride): [(13.0284961, 77.5710126), (13.0284963, 77.5710129), (13.0284964, 77.571013), (13.0286532, 77.5709275), (13.0287091, 77.5708699)] ...
Sample journey_hexes  (first ride): ['896014594d3ffff', '896014594d7ffff', '896014594c3ffff', '896014594c7ffff', '896014594cfffff'] ...

journey_coords lengths: min=1, max=2615, mean=264.5
journey_hexes  lengths: min=1, max=133, mean=20.3


/tmp/ipykernel_63683/3343135152.py:22: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [6]:

import heapq
import h3
import math
import numpy as np
import requests
from collections import defaultdict

def bearing(lat1, lon1, lat2, lon2):
    dlon = math.radians(lon2 - lon1)
    x = math.sin(dlon) * math.cos(math.radians(lat2))
    y = (math.cos(math.radians(lat1)) * math.sin(math.radians(lat2)) -
         math.sin(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.cos(dlon))
    return math.degrees(math.atan2(x, y)) % 360

def direction_compatible(path_a, path_b, threshold=90):
    bear_a = bearing(*path_a[0], *path_a[-1])
    bear_b = bearing(*path_b[0], *path_b[-1])
    diff = abs(bear_a - bear_b)
    diff = min(diff, 360 - diff)
    return diff < threshold

def path_similarity(h3_path_1, h3_path_2, coords_1, coords_2, k=2):
    if not direction_compatible(coords_1, coords_2, threshold=90):
        return 0.0
    
    set_1 = set()
    for h in h3_path_1:
        set_1.update(h3.grid_disk(h, k))
    
    set_2 = set()
    for h in h3_path_2:
        set_2.update(h3.grid_disk(h, k))
    
    cov_1 = sum(1 for h in h3_path_1 if h in set_2) / len(h3_path_1)
    cov_2 = sum(1 for h in h3_path_2 if h in set_1) / len(h3_path_2)
    
    return min(cov_1, cov_2)

def check_osrm_detour(A_curr, B_curr, A_end, B_end, threshold=30.0):
    def get_route_legs(coords):
        coords_str = ";".join([f"{lon},{lat}" for lat, lon in coords])
        url = f"http://127.0.0.1:5000/route/v1/driving/{coords_str}?overview=false"
        try:
            res = requests.get(url).json()
            if res.get("code") == "Ok":
                return [leg["distance"] for leg in res["routes"][0]["legs"]]
        except Exception:
            pass
        return None

    legs1 = get_route_legs([A_curr, B_curr, A_end, B_end])
    legs2 = get_route_legs([A_curr, B_curr, B_end, A_end])
    legsA = get_route_legs([A_curr, A_end])
    
    if not legs1 or not legs2 or not legsA:
        return False
        
    direct_A = legsA[0]
    direct_B = legs2[1]  
    
    if direct_A <= 0 or direct_B <= 0:
        return False
        
    dist1 = sum(legs1)
    dist2 = sum(legs2)
    
    if dist1 < dist2:
        detour_A = ((legs1[0] + legs1[1]) - direct_A) / direct_A * 100
        detour_B = ((legs1[1] + legs1[2]) - direct_B) / direct_B * 100
    else:
        detour_A = ((legs2[0] + legs2[1] + legs2[2]) - direct_A) / direct_A * 100
        detour_B = 0.0 
        
    if detour_A <= threshold and detour_B <= threshold:
        return True
    return False



# ── Rebuild ride_meta to include end coordinates, journey_coords and journey_hexes ──
ride_meta_sim = {}
for _, r in df2.iterrows():
    ride_meta_sim[r['rid']] = {
        'trip_end_time':  r['trip_end_time'],
        'trip_end_lat':   r['trip_end_lat'],
        'trip_end_lon':   r['trip_end_lon'],
        'journey_coords': r['journey_coords'],
        'journey_hexes':  r['journey_hexes']
    }

# df = df[df["rideStatus"]=="ON_RIDE"]

# ── Core structures ───────────────────────────────────────────────
active_heap    = []                
active_rides   = {}                
removed_rids   = set()             
hex_to_rids    = defaultdict(set)  
batched_rids   = set()
tried_rids   = {}
seq = 0
seq2 = 0
osrm_count=0

max_active       = 0
total_evicted    = 0
snapshot_interval = max(1, len(df) // 1000)
snapshots        = []

best_sim_scores = [np.nan] * len(df)
best_sim_rids   = [None] * len(df)
batchable       = [0] * len(df)

columns = df.columns.tolist()

for i, (idx, row) in enumerate(df.iterrows()):
    current_ts    = row['ts']
    rid           = row['rid']
    curr_hex      = row['curr_hex']
    current_ts_ns = current_ts.value
    status        = row['rideStatus']

    meta = ride_meta_sim.get(rid)
    if meta is None or not isinstance(meta.get('journey_hexes'), list) or not isinstance(meta.get('journey_coords'), list):
        continue

    end_time      = meta['trip_end_time']
    end_time_ns   = end_time.value

    while active_heap and active_heap[0][0] <= current_ts_ns:
        # print(f"Evicting rides at index {i} (ts={current_ts}) - active_heap size before eviction: {len(active_heap)}")
        _, _, evict_rid = heapq.heappop(active_heap)
        # if evict_rid in removed_rids:
        #     removed_rids.discard(evict_rid)
        #     continue
        if evict_rid in active_rides:
            evict_hex = active_rides[evict_rid].get('curr_hex')
            if evict_hex and evict_rid in hex_to_rids[evict_hex]:
                hex_to_rids[evict_hex].discard(evict_rid)
                if not hex_to_rids[evict_hex]:
                    del hex_to_rids[evict_hex]
            del active_rides[evict_rid]
            total_evicted += 1

    is_new_ride = rid not in active_rides

    if not is_new_ride:
        old_hex = active_rides[rid].get('curr_hex')
        if old_hex != curr_hex:
            if old_hex and rid in hex_to_rids[old_hex]:
                hex_to_rids[old_hex].discard(rid)
                if not hex_to_rids[old_hex]:
                    del hex_to_rids[old_hex]
            hex_to_rids[curr_hex].add(rid)
        removed_rids.add(rid)

        ride_data = {c: row[c] for c in columns}
        ride_data['journey_hexes'] = meta['journey_hexes']
        ride_data['journey_coords'] = meta['journey_coords']
        ride_data['trip_end_lat'] = meta['trip_end_lat']
        ride_data['trip_end_lon'] = meta['trip_end_lon']
    
        active_rides[rid] = ride_data
    
        # seq += 1
        # heapq.heappush(active_heap, (end_time_ns, seq, rid))
    
    is_not_batched_ride = rid not in batched_rids

    is_tried_ride = rid in tried_rids

    if is_tried_ride and current_ts_ns - tried_rids[rid] < 0.1 * 1e9:
        is_tried_ride = True
        

    if is_new_ride and not is_tried_ride and is_not_batched_ride and status == "ON_PICKUP":

        tried_rids[rid] = current_ts_ns
        disk_hexes = h3.grid_disk(curr_hex, 1)
        neighbor_rids = []
        for h in disk_hexes:
            if h in hex_to_rids:
                neighbor_rids.extend(hex_to_rids[h])

        B_hexes = meta['journey_hexes']
        B_coords = meta['journey_coords']
        
        try:
            b_start_idx = B_hexes.index(curr_hex)
        except ValueError:
            b_start_idx = 0
            
        rem_B_hexes = B_hexes[b_start_idx:]
        rem_B_coords = [(row['lat'], row['lon']), B_coords[-1]] if len(B_coords) > 0 else []

        best_score = 0.0
        best_rid = None

        if len(neighbor_rids) > 0 and len(rem_B_hexes) > 0 and len(rem_B_coords) >= 2:
            for n_rid in neighbor_rids:
                if n_rid in batched_rids:
                    continue
                ar = active_rides[n_rid]
                
                A_hexes = ar['journey_hexes']
                A_coords = ar['journey_coords']
                A_curr_hex = ar['curr_hex']
                
                try:
                    a_start_idx = A_hexes.index(A_curr_hex)
                except ValueError:
                    a_start_idx = 0
                    
                rem_A_hexes = A_hexes[a_start_idx:]
                rem_A_coords = [(ar['lat'], ar['lon']), A_coords[-1]] if len(A_coords) > 0 else []

                if len(rem_A_hexes) > 0 and len(rem_A_coords) >= 2:
                    sim = path_similarity(rem_A_hexes, rem_B_hexes, rem_A_coords, rem_B_coords, k=2)
                    if sim > best_score:
                        best_score = sim
                        best_rid = n_rid
                        
        best_sim_scores[i] = best_score
        
        if best_score > 0.0 and best_rid is not None:
            best_sim_rids[i] = best_rid
            
            ar = active_rides[best_rid]
            A_curr = (ar['lat'], ar['lon'])
            B_curr = (row['lat'], row['lon'])
            
            A_end = (ar['trip_end_lat'], ar['trip_end_lon'])
            B_end = (meta['trip_end_lat'], meta['trip_end_lon'])
            
            is_batchable = check_osrm_detour(A_curr, B_curr, A_end, B_end, threshold=30.0)
            osrm_count+=1
            if is_batchable:
                batchable[i] = 1
                batched_rids.add(best_rid)
                batched_rids.add(rid)


    if is_new_ride and status == "ON_RIDE":
        hex_to_rids[curr_hex].add(rid)

        ride_data = {c: row[c] for c in columns}
        ride_data['journey_hexes'] = meta['journey_hexes']
        ride_data['journey_coords'] = meta['journey_coords']
        ride_data['trip_end_lat'] = meta['trip_end_lat']
        ride_data['trip_end_lon'] = meta['trip_end_lon']
    
        active_rides[rid] = ride_data
        seq += 1
        heapq.heappush(active_heap, (end_time_ns, seq, rid))
        print(seq,osrm_count)
        
    
    n_active = len(active_rides)
    if n_active > max_active:
        max_active = n_active
        # print(f"New peak active rides: {max_active} at index {i} (ts={current_ts})")

    

df['best_sim_score'] = best_sim_scores
df['best_sim_rid'] = best_sim_rids
df['batchable'] = batchable

print(f"✓ Done")
print(f"  Peak active rides         : {max_active}")
print(f"  Total evicted             : {total_evicted}")
print(f"  Still active (end)        : {len(active_rides)}")

batchable_df = df[df['batchable'] == 1]
print(f"  Batchable matches found   : {len(batchable_df)}")

print(len(batched_rids))
# if len(batchable_df) > 0:
#     print("\nSample batchable matches:")
#     res = batchable_df[['ts', 'rid', 'curr_hex', 'best_sim_score', 'best_sim_rid', 'batchable']].head(10)
#     print(res.to_string(index=False))

print(f"  OSRM checks performed     : {osrm_count}")

1 0
2 0
3 0
4 0
5 0
6 0
7 0
8 0
9 0
10 0
11 0
12 0
13 0
14 0
15 0
16 0
17 0
18 0
19 0
20 0
21 0
22 0
23 0
24 0
25 0
26 0
27 0
28 0
29 0
30 0
31 0
32 0
33 0
34 0
35 0
36 0
37 0
38 0
39 0
40 0
41 0
42 0
43 0
44 0
45 0
46 0
47 0
48 0
49 0
50 0
51 0
52 0
53 0
54 0
55 0
56 0
57 0
58 1
59 1
60 1
61 1
62 1
63 1
64 1
65 1
66 1
67 1
68 1
69 1
70 1
71 1
72 1
73 1
74 1
75 1
76 1
77 1
78 2
79 2
80 2
81 2
82 2
83 2
84 2
85 2
86 2
87 2
88 3
89 3
90 3
91 5
92 6
93 6
94 6
95 6
96 7
97 7
98 7
99 7
100 7
101 7
102 7
103 7
104 7
105 7
106 7
107 7
108 8
109 8
110 8
111 8
112 8
113 8
114 8
115 9
116 9
117 9
118 9
119 9
120 9
121 9
122 9
123 9
124 9
125 9
126 9
127 10
128 10
129 10
130 11
131 11
132 11
133 11
134 11
135 11
136 11
137 11
138 11
139 11
140 13
141 13
142 13
143 15
144 15
145 15
146 15
147 15
148 15
149 15
150 15
151 15
152 15
153 16
154 17
155 18
156 18
157 18
158 18
159 18
160 18
161 18
162 18
163 18
164 18
165 20
166 20
167 20
168 20
169 20
170 20
171 20
172 21
173 21
174 21
175 21
176 21
17

In [7]:
check = batchable_df["rid"].drop_duplicates().shape[0]
print(f"Unique batchable rids: {check}")

Unique batchable rids: 2348


In [8]:
len(batched_rids)

4696

In [9]:
df2=df[df["rideStatus"]=="ON_PICKUP"]

In [10]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 831202 entries, 0 to 4637533
Data columns (total 9 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   rid             831202 non-null  object        
 1   ts              831202 non-null  datetime64[ns]
 2   lat             831202 non-null  float64       
 3   lon             831202 non-null  float64       
 4   rideStatus      831202 non-null  object        
 5   curr_hex        831202 non-null  object        
 6   best_sim_score  14390 non-null   float64       
 7   best_sim_rid    8107 non-null    object        
 8   batchable       831202 non-null  int64         
dtypes: datetime64[ns](1), float64(3), int64(1), object(4)
memory usage: 63.4+ MB
